In [1]:
!pip install fugashi unidic-lite pandas scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [9]:
"""
phase1
人間が解釈できる特徴量のみで分類器を作る
->n-gram除去
"""
import glob
import os
import re
from collections import Counter
import numpy as np
import pandas as pd
from fugashi import Tagger
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# 形態素解析用の共通インスタンスと品詞定義
tagger = Tagger()
FUNCTION_POS = {"助詞", "助動詞", "接続詞", "連体詞"}
CONTENT_POS = {"名詞", "動詞", "形容詞", "形状詞", "副詞", "感動詞"}


def analyze_text_files(folder_path, label):
    """指定されたフォルダ内の全テキストファイルを読み込み、形態素解析と各種特徴量（Hapax, Yules_K等）を抽出する"""
    # 数値順にソートしてファイルを取得
    search_path = os.path.join(folder_path, "*.txt")
    filepaths = sorted(
        glob.glob(search_path),
        key=lambda x: int(re.search(r"\d+", os.path.basename(x)).group())
        if re.search(r"\d+", os.path.basename(x))
        else 0,
    )

    files = []
    tokenized_texts = []
    content_texts = []
    function_texts = []
    function_counts = []
    content_counts = []
    function_pers = []
    content_pers = []
    feature_data = []

    for filepath in filepaths:
        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()
            files.append((filepath, text))

        pos_counter = Counter()
        particle_counter = Counter()
        tokens, content_words, function_words, all_words = [], [], [], []
        word_count = content_count = function_count = 0

        # 記号のカウントと文長計算
        comma_count = text.count("、")
        period_count = text.count("。")
        exclamation_count = text.count("！")
        question_count = text.count("？")

        sentences = re.split(r"[。！？]", text)
        sentence_lengths = [len(s.strip()) for s in sentences if s.strip()]
        avg_sentence_length = (
            np.mean(sentence_lengths) if sentence_lengths else 0
        )
        
        # 文字種特徴
        total_chars = len(text)
        
        hiragana_count = len(re.findall(r"[ぁ-ん]", text))
        katakana_count = len(re.findall(r"[ァ-ン]", text))
        kanji_count = len(re.findall(r"[一-龥]", text))
        alnum_count = len(re.findall(r"[a-zA-Z0-9]", text))
        
        hiragana_ratio = hiragana_count / total_chars if total_chars else 0
        katakana_ratio = katakana_count / total_chars if total_chars else 0
        kanji_ratio = kanji_count / total_chars if total_chars else 0
        alnum_ratio = alnum_count / total_chars if total_chars else 0
        
        # 文長特徴
        sentence_std = (
            np.std(sentence_lengths)
            if sentence_lengths else 0
        )
        
        max_sentence_length = (
            max(sentence_lengths)
            if sentence_lengths else 0
        )

        # 形態素解析
        for word in tagger(text):
            pos = word.feature.pos1
            surface = word.surface

            if pos == "補助記号":
                continue
            if pos == "助詞":
                particle_counter[surface] += 1

            tokens.append(surface)
            all_words.append(surface)
            pos_counter[pos] += 1
            word_count += 1

            if pos in FUNCTION_POS:
                function_words.append(surface)
                function_count += 1
            elif pos in CONTENT_POS:
                content_words.append(surface)
                content_count += 1

        # テキストの結合
        tokenized_texts.append(" ".join(tokens))
        function_texts.append(" ".join(function_words))
        content_texts.append(" ".join(content_words))
        function_counts.append(function_count)
        content_counts.append(content_count)
        
        # 基本割合・指標計算
        if word_count > 0:
            f_per = (function_count / word_count) * 100
            c_per = (content_count / word_count) * 100
            comma_ratio = comma_count / word_count
            period_ratio = period_count / word_count
            ttr = len(set(all_words)) / word_count
            # ===== Hapax / Yule（全・機能・内容）=====
            hapax_all, yule_all = hapax_yule(all_words, word_count)
            hapax_func, yule_func = hapax_yule(function_words, function_count)
            hapax_cont, yule_cont = hapax_yule(content_words, content_count)

        function_pers.append(f_per)
        content_pers.append(c_per)

        total_particle = sum(particle_counter.values())
        particle_ratio = {
            f"助詞_{k}": (v / total_particle if total_particle > 0 else 0)
            for k, v in particle_counter.items()
        }
        pos_ratio = {
            pos: (count / word_count) * 100 for pos, count in pos_counter.items()
        }

        feature_data.append(
            {
                "filename": os.path.basename(filepath),
                "label": label,
                
                "総単語数": word_count,
                "機能語率": f_per,
                "内容語率": c_per,
                "異なり語数": len(set(all_words)),
                "TTR": ttr,
                
                "ひらがな率": hiragana_ratio,
                "カタカナ率": katakana_ratio,
                "漢字率": kanji_ratio,
                "英数字率": alnum_ratio,
                
                "文長標準偏差": sentence_std,
                "最大文長": max_sentence_length,
                
                "Hapax率_all": hapax_all,
                "Yules_K_all": yule_all,
                
                "Hapax率_func": hapax_func,
                "Yules_K_func": yule_func,
                
                "Hapax率_cont": hapax_cont,
                "Yules_K_cont": yule_cont,
                
                "平均文長": avg_sentence_length,
                
                "名詞割合": pos_ratio.get("名詞", 0),
                "動詞割合": pos_ratio.get("動詞", 0),
                "形容詞割合": pos_ratio.get("形容詞", 0),
                "形状詞割合": pos_ratio.get("形状詞", 0),
                "助詞割合": pos_ratio.get("助詞", 0),
                "助動詞割合": pos_ratio.get("助動詞", 0),
                "副詞割合": pos_ratio.get("副詞", 0),
                "連体詞割合": pos_ratio.get("連体詞", 0),
                "接続詞割合": pos_ratio.get("接続詞", 0),
                "感動詞割合": pos_ratio.get("感動詞", 0),
                
                **particle_ratio,
                
                "読点頻度": comma_ratio,
                "句点頻度": period_ratio,
                "感嘆符数": exclamation_count,
                "疑問符数": question_count,
            }
        )

    return {
        "files": files,
        "tokenized_texts": tokenized_texts,
        "content_texts": content_texts,
        "function_texts": function_texts,
        "function_counts": function_counts,
        "content_counts": content_counts,
        "function_pers": function_pers,
        "content_pers": content_pers,
        "feature_data": feature_data,
    }

def hapax_yule(tokens, word_count):
    if word_count == 0:
        return 0, 0

    freq = Counter(tokens)

    # Hapax率
    hapax_count = sum(1 for v in freq.values() if v == 1)
    hapax_ratio = hapax_count / word_count

    # Yule's K
    freq_counter = Counter(freq.values())
    m1 = word_count
    m2 = sum((f ** 2) * c for f, c in freq_counter.items())

    yules_k = 10000 * ((m2 - m1) / (m1 * m1)) if m1 > 0 else 0

    return hapax_ratio, yules_k

def jpn_word_ngram_analyzer(text):
    """半角スペースを挟ずに日本語の単語N-gram（2-gram, 3-gram）を生成する"""
    tokens = text.split()
    ngrams = []
    for n in [2, 3]:
        for i in range(len(tokens) - n + 1):
            ngrams.append("".join(tokens[i : i + n]))
    return ngrams


def print_analysis_results(
    data,
    X_content,
    content_features,
    Y_function,
    function_features,
    X_function_ngram,
    function_ngram_features,
    X_content_ngram,
    content_ngram_features,
    X_char_ngram,
    char_ngram_features,
    top_k=30,
):
    """解析結果を画面にわかりやすく表示する"""
    for i, (filepath, text) in enumerate(data["files"]):
        function_ngram_values = X_function_ngram.toarray()[i]
        content_ngram_values = X_content_ngram.toarray()[i]
        char_ngram_values = X_char_ngram.toarray()[i]

        # 各種N-gram上位の抽出
        function_ngram_top = dict(
            sorted(
                [
                    (f, int(v))
                    for f, v in zip(
                        function_ngram_features, function_ngram_values
                    )
                ],
                key=lambda x: x[1],
                reverse=True,
            )[:top_k]
        )
        content_ngram_top = dict(
            sorted(
                [
                    (f, int(v))
                    for f, v in zip(content_ngram_features, content_ngram_values)
                ],
                key=lambda x: x[1],
                reverse=True,
            )[:top_k]
        )
        char_top = dict(
            sorted(
                [
                    (f, int(v))
                    for f, v in zip(char_ngram_features, char_ngram_values)
                ],
                key=lambda x: x[1],
                reverse=True,
            )[:top_k]
        )

        data["feature_data"][i].update(
            {
                "function_ngram_top": function_ngram_top,
                "content_ngram_top": content_ngram_top,
                "char_ngram_top": char_top,
            }
        )

        print(f"\n=== ファイル：{filepath} ===")
        for key, value in data["feature_data"][i].items():
            if isinstance(value, float):
                print(f"{key}：{value:.4f}")
            else:
                print(f"{key}：{value}")

        print("\n=== 機能語 ===")
        print(data["function_texts"][i])
        print(f"カウント数：{data['function_counts'][i]}個")
        print(f"割合：{data['function_pers'][i]:.2f}%")
        for word, score in zip(function_features, Y_function.toarray()[i]):
            if score > 0:
                print(f"{word} {score:.4f}")

        print("\n=== 内容語 ===")
        print(data["content_texts"][i])
        print(f"カウント数：{data['content_counts'][i]}個")
        print(f"割合：{data['content_pers'][i]:.2f}%")
        for word, score in zip(content_features, X_content.toarray()[i]):
            if score > 0:
                print(f"{word} {score:.4f}")


if __name__ == "__main__":
    # フォルダパスを指定してテキスト解析を実行
    human_data = analyze_text_files(
        "train_Human_text",
        "Human",
    )
    
    ai_data = analyze_text_files(
        "train_AI_text",
        "AI"
    )

        
    all_features = (
        human_data["feature_data"]
        + ai_data["feature_data"]
    )

    df = pd.DataFrame(all_features)

    # df = df.sample(
    #     frac=1,
    #     random_state=42
    # ).reset_index(drop=True)

    df.to_csv(
        "stylometry_features_train.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(df.head())
    print(f"総データ数：{len(df)}")

            filename  label  総単語数       機能語率       内容語率  異なり語数       TTR  \
0  Human_text_01.txt  Human   243  47.736626  51.440329     94  0.386831   
1  Human_text_02.txt  Human   279  41.577061  54.121864    134  0.480287   
2  Human_text_03.txt  Human   319  46.708464  48.902821    119  0.373041   
3  Human_text_04.txt  Human   281  49.110320  48.042705    135  0.480427   
4  Human_text_05.txt  Human   313  44.728435  50.159744    126  0.402556   

      ひらがな率     カタカナ率       漢字率  ...  助詞_のみ  助詞_しも  助詞_しか  助詞_すら  助詞_より  \
0  0.417021  0.268085  0.219149  ...    NaN    NaN    NaN    NaN    NaN   
1  0.492000  0.070000  0.296000  ...    NaN    NaN    NaN    NaN    NaN   
2  0.569790  0.053537  0.315488  ...    NaN    NaN    NaN    NaN    NaN   
3  0.544304  0.025316  0.356540  ...    NaN    NaN    NaN    NaN    NaN   
4  0.579365  0.039683  0.321429  ...    NaN    NaN    NaN    NaN    NaN   

   助詞_こそ  助詞_つつ  助詞_くらい  助詞_かしら  助詞_ずつ  
0    NaN    NaN     NaN     NaN    NaN  
1    NaN  

In [10]:
"""
phase1
人間が解釈できる特徴量のみで分類器を作る
->n-gram除去
"""
import glob
import os
import re
from collections import Counter
import numpy as np
import pandas as pd
from fugashi import Tagger
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# 形態素解析用の共通インスタンスと品詞定義
tagger = Tagger()
FUNCTION_POS = {"助詞", "助動詞", "接続詞", "連体詞"}
CONTENT_POS = {"名詞", "動詞", "形容詞", "形状詞", "副詞", "感動詞"}


def analyze_text_files(folder_path, label):
    """指定されたフォルダ内の全テキストファイルを読み込み、形態素解析と各種特徴量（Hapax, Yules_K等）を抽出する"""
    # 数値順にソートしてファイルを取得
    search_path = os.path.join(folder_path, "*.txt")
    filepaths = sorted(
        glob.glob(search_path),
        key=lambda x: int(re.search(r"\d+", os.path.basename(x)).group())
        if re.search(r"\d+", os.path.basename(x))
        else 0,
    )

    files = []
    tokenized_texts = []
    content_texts = []
    function_texts = []
    function_counts = []
    content_counts = []
    function_pers = []
    content_pers = []
    feature_data = []

    for filepath in filepaths:
        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()
            files.append((filepath, text))

        pos_counter = Counter()
        particle_counter = Counter()
        tokens, content_words, function_words, all_words = [], [], [], []
        word_count = content_count = function_count = 0

        # 記号のカウントと文長計算
        comma_count = text.count("、")
        period_count = text.count("。")
        exclamation_count = text.count("！")
        question_count = text.count("？")

        sentences = re.split(r"[。！？]", text)
        sentence_lengths = [len(s.strip()) for s in sentences if s.strip()]
        avg_sentence_length = (
            np.mean(sentence_lengths) if sentence_lengths else 0
        )
        
        # 文字種特徴
        total_chars = len(text)
        
        hiragana_count = len(re.findall(r"[ぁ-ん]", text))
        katakana_count = len(re.findall(r"[ァ-ン]", text))
        kanji_count = len(re.findall(r"[一-龥]", text))
        alnum_count = len(re.findall(r"[a-zA-Z0-9]", text))
        
        hiragana_ratio = hiragana_count / total_chars if total_chars else 0
        katakana_ratio = katakana_count / total_chars if total_chars else 0
        kanji_ratio = kanji_count / total_chars if total_chars else 0
        alnum_ratio = alnum_count / total_chars if total_chars else 0
        
        # 文長特徴
        sentence_std = (
            np.std(sentence_lengths)
            if sentence_lengths else 0
        )
        
        max_sentence_length = (
            max(sentence_lengths)
            if sentence_lengths else 0
        )

        # 形態素解析
        for word in tagger(text):
            pos = word.feature.pos1
            surface = word.surface

            if pos == "補助記号":
                continue
            if pos == "助詞":
                particle_counter[surface] += 1

            tokens.append(surface)
            all_words.append(surface)
            pos_counter[pos] += 1
            word_count += 1

            if pos in FUNCTION_POS:
                function_words.append(surface)
                function_count += 1
            elif pos in CONTENT_POS:
                content_words.append(surface)
                content_count += 1

        # テキストの結合
        tokenized_texts.append(" ".join(tokens))
        function_texts.append(" ".join(function_words))
        content_texts.append(" ".join(content_words))
        function_counts.append(function_count)
        content_counts.append(content_count)
        
        # 基本割合・指標計算
        if word_count > 0:
            f_per = (function_count / word_count) * 100
            c_per = (content_count / word_count) * 100
            comma_ratio = comma_count / word_count
            period_ratio = period_count / word_count
            ttr = len(set(all_words)) / word_count
            # ===== Hapax / Yule（全・機能・内容）=====
            hapax_all, yule_all = hapax_yule(all_words, word_count)
            hapax_func, yule_func = hapax_yule(function_words, function_count)
            hapax_cont, yule_cont = hapax_yule(content_words, content_count)

        function_pers.append(f_per)
        content_pers.append(c_per)

        total_particle = sum(particle_counter.values())
        particle_ratio = {
            f"助詞_{k}": (v / total_particle if total_particle > 0 else 0)
            for k, v in particle_counter.items()
        }
        pos_ratio = {
            pos: (count / word_count) * 100 for pos, count in pos_counter.items()
        }

        feature_data.append(
            {
                "filename": os.path.basename(filepath),
                "label": label,
                
                "総単語数": word_count,
                "機能語率": f_per,
                "内容語率": c_per,
                "異なり語数": len(set(all_words)),
                "TTR": ttr,
                
                "ひらがな率": hiragana_ratio,
                "カタカナ率": katakana_ratio,
                "漢字率": kanji_ratio,
                "英数字率": alnum_ratio,
                
                "文長標準偏差": sentence_std,
                "最大文長": max_sentence_length,
                
                "Hapax率_all": hapax_all,
                "Yules_K_all": yule_all,
                
                "Hapax率_func": hapax_func,
                "Yules_K_func": yule_func,
                
                "Hapax率_cont": hapax_cont,
                "Yules_K_cont": yule_cont,
                
                "平均文長": avg_sentence_length,
                
                "名詞割合": pos_ratio.get("名詞", 0),
                "動詞割合": pos_ratio.get("動詞", 0),
                "形容詞割合": pos_ratio.get("形容詞", 0),
                "形状詞割合": pos_ratio.get("形状詞", 0),
                "助詞割合": pos_ratio.get("助詞", 0),
                "助動詞割合": pos_ratio.get("助動詞", 0),
                "副詞割合": pos_ratio.get("副詞", 0),
                "連体詞割合": pos_ratio.get("連体詞", 0),
                "接続詞割合": pos_ratio.get("接続詞", 0),
                "感動詞割合": pos_ratio.get("感動詞", 0),
                
                **particle_ratio,
                
                "読点頻度": comma_ratio,
                "句点頻度": period_ratio,
                "感嘆符数": exclamation_count,
                "疑問符数": question_count,
            }
        )

    return {
        "files": files,
        "tokenized_texts": tokenized_texts,
        "content_texts": content_texts,
        "function_texts": function_texts,
        "function_counts": function_counts,
        "content_counts": content_counts,
        "function_pers": function_pers,
        "content_pers": content_pers,
        "feature_data": feature_data,
    }

def hapax_yule(tokens, word_count):
    if word_count == 0:
        return 0, 0

    freq = Counter(tokens)

    # Hapax率
    hapax_count = sum(1 for v in freq.values() if v == 1)
    hapax_ratio = hapax_count / word_count

    # Yule's K
    freq_counter = Counter(freq.values())
    m1 = word_count
    m2 = sum((f ** 2) * c for f, c in freq_counter.items())

    yules_k = 10000 * ((m2 - m1) / (m1 * m1)) if m1 > 0 else 0

    return hapax_ratio, yules_k

def jpn_word_ngram_analyzer(text):
    """半角スペースを挟ずに日本語の単語N-gram（2-gram, 3-gram）を生成する"""
    tokens = text.split()
    ngrams = []
    for n in [2, 3]:
        for i in range(len(tokens) - n + 1):
            ngrams.append("".join(tokens[i : i + n]))
    return ngrams


def print_analysis_results(
    data,
    X_content,
    content_features,
    Y_function,
    function_features,
    X_function_ngram,
    function_ngram_features,
    X_content_ngram,
    content_ngram_features,
    X_char_ngram,
    char_ngram_features,
    top_k=30,
):
    """解析結果を画面にわかりやすく表示する"""
    for i, (filepath, text) in enumerate(data["files"]):
        function_ngram_values = X_function_ngram.toarray()[i]
        content_ngram_values = X_content_ngram.toarray()[i]
        char_ngram_values = X_char_ngram.toarray()[i]

        # 各種N-gram上位の抽出
        function_ngram_top = dict(
            sorted(
                [
                    (f, int(v))
                    for f, v in zip(
                        function_ngram_features, function_ngram_values
                    )
                ],
                key=lambda x: x[1],
                reverse=True,
            )[:top_k]
        )
        content_ngram_top = dict(
            sorted(
                [
                    (f, int(v))
                    for f, v in zip(content_ngram_features, content_ngram_values)
                ],
                key=lambda x: x[1],
                reverse=True,
            )[:top_k]
        )
        char_top = dict(
            sorted(
                [
                    (f, int(v))
                    for f, v in zip(char_ngram_features, char_ngram_values)
                ],
                key=lambda x: x[1],
                reverse=True,
            )[:top_k]
        )

        data["feature_data"][i].update(
            {
                "function_ngram_top": function_ngram_top,
                "content_ngram_top": content_ngram_top,
                "char_ngram_top": char_top,
            }
        )

        print(f"\n=== ファイル：{filepath} ===")
        for key, value in data["feature_data"][i].items():
            if isinstance(value, float):
                print(f"{key}：{value:.4f}")
            else:
                print(f"{key}：{value}")

        print("\n=== 機能語 ===")
        print(data["function_texts"][i])
        print(f"カウント数：{data['function_counts'][i]}個")
        print(f"割合：{data['function_pers'][i]:.2f}%")
        for word, score in zip(function_features, Y_function.toarray()[i]):
            if score > 0:
                print(f"{word} {score:.4f}")

        print("\n=== 内容語 ===")
        print(data["content_texts"][i])
        print(f"カウント数：{data['content_counts'][i]}個")
        print(f"割合：{data['content_pers'][i]:.2f}%")
        for word, score in zip(content_features, X_content.toarray()[i]):
            if score > 0:
                print(f"{word} {score:.4f}")


if __name__ == "__main__":
    # フォルダパスを指定してテキスト解析を実行
    human_data = analyze_text_files(
        "test_Human_text",
        "Human",
    )
    
    ai_data = analyze_text_files(
        "test_AI_text",
        "AI"
    )

        
    all_features = (
        human_data["feature_data"]
        + ai_data["feature_data"]
    )

    df = pd.DataFrame(all_features)

    # df = df.sample(
    #     frac=1,
    #     random_state=42
    # ).reset_index(drop=True)

    df.to_csv(
        "stylometry_features_test.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(df.head())
    print(f"総データ数：{len(df)}")

                 filename  label  総単語数       機能語率       内容語率  異なり語数       TTR  \
0  test_Human_text_01.txt  Human   289  42.214533  53.287197    139  0.480969   
1  test_Human_text_02.txt  Human   351  48.148148  47.293447    143  0.407407   
2  test_Human_text_03.txt  Human   311  45.980707  47.588424    144  0.463023   
3  test_Human_text_04.txt  Human   321  46.105919  46.105919    139  0.433022   
4  test_Human_text_05.txt  Human   272  33.823529  58.455882    131  0.481618   

      ひらがな率     カタカナ率       漢字率  ...      読点頻度      句点頻度  感嘆符数  疑問符数  \
0  0.443820  0.168539  0.312734  ...  0.048443  0.031142     0     0   
1  0.531952  0.000000  0.371330  ...  0.056980  0.037037     0     0   
2  0.521484  0.011719  0.408203  ...  0.057878  0.028939     0     0   
3  0.591682  0.007561  0.345936  ...  0.049844  0.034268     0     0   
4  0.332636  0.110879  0.414226  ...  0.062500  0.036765     0     0   

      助詞_まで      助詞_か     助詞_たり  助詞_だけ  助詞_ながら  助詞_より  
0       NaN       NaN   